In [2]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/shopping_trends.csv")
X = pd.read_csv("../data/clustering_features.csv")
reference_labels = np.load("../data/reference_labels.npy")

profile = X.copy()
profile["Cluster"] = reference_labels

print("Shape:", profile.shape)
profile.head()

Shape: (3900, 9)


,Age,Purchase Amount (USD),Review Rating,Previous Purchases,Purchase Frequency,Is_Subscribed,Log_Annual_Spend,Log_Purchase_Engagement,Cluster
0,55,53,3.1,14,26,1,7.229114,5.899897,1
1,19,64,3.1,2,26,1,7.417580,3.970292,1
2,50,73,3.1,23,52,1,8.241967,7.087574,1
3,21,90,3.5,49,52,1,8.451267,7.843456,1
4,45,49,2.7,31,1,1,3.912023,3.465736,0


In [4]:
business_value = (
    profile
    .groupby("Cluster")[
        [
            "Purchase Amount (USD)",
            "Previous Purchases",
            "Purchase Frequency",
        ]
    ]
    .mean()
    .round(2)
)

business_value

,Purchase Amount (USD),Previous Purchases,Purchase Frequency
Cluster,,,
0,58.44,23.89,4.37
1,61.14,26.87,31.07


In [5]:
comparison = pd.DataFrame({
    "Metric": [
        "Purchase Amount",
        "Previous Purchases",
        "Purchase Frequency"
    ],
    "Cluster 0": [
        business_value.loc[0, "Purchase Amount (USD)"],
        business_value.loc[0, "Previous Purchases"],
        business_value.loc[0, "Purchase Frequency"]
    ],
    "Cluster 1": [
        business_value.loc[1, "Purchase Amount (USD)"],
        business_value.loc[1, "Previous Purchases"],
        business_value.loc[1, "Purchase Frequency"]
    ]
})

comparison["Relative Difference (%)"] = (
    (comparison["Cluster 1"] - comparison["Cluster 0"])
    / comparison["Cluster 0"] * 100
).round(2)

comparison

,Metric,Cluster 0,Cluster 1,Relative Difference (%)
0,Purchase Amount,58.44,61.14,4.62
1,Previous Purchases,23.89,26.87,12.47
2,Purchase Frequency,4.37,31.07,610.98


In [6]:
profile["Estimated_Annual_Spend"] = (
    profile["Purchase Amount (USD)"]
    * profile["Purchase Frequency"]
)

annual_spend = (
    profile
    .groupby("Cluster")["Estimated_Annual_Spend"]
    .mean()
    .round(2)
)

annual_spend

Cluster
0     238.94
1    1869.01
Name: Estimated_Annual_Spend, dtype: float64

In [7]:
annual_spend_comparison = pd.DataFrame({
    "Cluster 0": [annual_spend.loc[0]],
    "Cluster 1": [annual_spend.loc[1]]
}, index=["Estimated Annual Spend"])

annual_spend_comparison["Relative Difference (%)"] = (
    (annual_spend_comparison["Cluster 1"]
     - annual_spend_comparison["Cluster 0"])
    / annual_spend_comparison["Cluster 0"] * 100
).round(2)

annual_spend_comparison

,Cluster 0,Cluster 1,Relative Difference (%)
Estimated Annual Spend,238.94,1869.01,682.21


The segmentation primarily differentiates customers by purchase intensity rather than by transaction value or demographic characteristics. Cluster 1 exhibits substantially higher purchase frequency and therefore a higher derived annual spending estimate, while average purchase value remains relatively similar across segments. However, the relatively low silhouette score indicates limited cluster separation, and the annual spending metric should be interpreted as a derived estimate rather than observed longitudinal revenue.